# DWBDA Assignments

**Name:** Somya Bharti  
**Registration No.:** 202400480  
**Subject:** DWBDA (Data Warehouse and Business Data Analytics)

This notebook contains three assignments:
1. Market Basket Analysis using an uncommon eco-refill retail transaction dataset.
2. MapReduce-style calculation of students in grade categories for every subject.
3. Spark aggregation of specialty craft and hobby products by product line.

**Dataset note:** Each included CSV contains at least 300 records. The datasets are structured practice datasets created for this assignment and are not claimed to be downloaded Kaggle originals. If the instructor strictly requires Kaggle provenance, replace the relevant CSV with a downloaded Kaggle file and record its URL and license in the README.

## Common imports and setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("data")
pd.set_option("display.max_columns", None)


# Assignment 1: Market Basket Analysis (MBA)

**Objective:** Identify frequently purchased item combinations and generate association rules.

**Dataset theme:** Eco-friendly household and refill-store products, with 350 transactions and 20 product types.

The transaction dataset contains one row per item purchased in a transaction. We use Apriori and association-rule metrics:
- **Support:** Frequency of an itemset in all transactions.
- **Confidence:** Probability of buying the consequent when the antecedent is purchased.
- **Lift:** Strength of association compared with random co-occurrence.

In [ ]:
# Install dependency in the notebook environment if needed:
# %pip install mlxtend

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

mba = pd.read_csv(DATA_DIR / "grocery_transactions.csv")
mba.head()


In [ ]:
# Convert rows into transaction lists
transactions = mba.groupby("TransactionID")["Item"].apply(list).tolist()

te = TransactionEncoder()
encoded = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(encoded, columns=te.columns_)

print("Number of transactions:", len(transactions))
print("Number of unique items:", len(te.columns_))
basket_df.head()


In [ ]:
# Find frequent itemsets
frequent_itemsets = apriori(
    basket_df,
    min_support=0.10,
    use_colnames=True
).sort_values("support", ascending=False)

frequent_itemsets.head(10)


In [ ]:
# Generate association rules
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.30
)

rules = rules.sort_values(
    ["lift", "confidence"],
    ascending=False
)

rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)


In [ ]:
# Visualize the most frequent individual items
item_frequency = basket_df.sum().sort_values(ascending=False).head(10)

item_frequency.plot(kind="bar", figsize=(10, 5), title="Top 10 Items by Transaction Frequency")
plt.xlabel("Item")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Conclusion:** Market Basket Analysis identifies items that frequently occur together. The generated support, confidence, and lift values can help retailers design product bundles, recommendations, and shelf-placement strategies.

# Assignment 2: MapReduce Framework

**Objective:** Calculate the number of students in each category (S, A, B, C, D, E, F) for every subject using the previous semester's final examination results.

The implementation simulates MapReduce in Python:
- **Mapper:** Emits `(subject, grade)` key-value pairs.
- **Reducer:** Counts the grades for each subject.

In [ ]:
results = pd.read_csv(DATA_DIR / "previous_semester_results.csv")
results.head()


In [ ]:
from collections import defaultdict, Counter

subjects = [col for col in results.columns if col != "Student_ID"]

# Mapper phase
mapped_pairs = []
for _, row in results.iterrows():
    for subject in subjects:
        mapped_pairs.append((subject, row[subject]))

print("Sample mapper output:", mapped_pairs[:10])


In [ ]:
# Shuffle and sort phase
grouped = defaultdict(list)
for subject, grade in mapped_pairs:
    grouped[subject].append(grade)

# Reducer phase
grade_order = ["S", "A", "B", "C", "D", "E", "F"]
reduced_output = {}

for subject, grades_for_subject in grouped.items():
    counts = Counter(grades_for_subject)
    reduced_output[subject] = {
        grade: counts.get(grade, 0) for grade in grade_order
    }

grade_summary = pd.DataFrame.from_dict(reduced_output, orient="index")
grade_summary = grade_summary[grade_order]

grade_summary


In [ ]:
grade_summary.plot(kind="bar", figsize=(12, 6))
plt.title("Student Count by Grade Category for Each Subject")
plt.xlabel("Subject")
plt.ylabel("Number of Students")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Grade")
plt.tight_layout()
plt.show()


**Conclusion:** The MapReduce-style workflow successfully groups student grades by subject and counts the number of students in every category from S to F.

# Assignment 3: Spark – Products Sold in Each Category

**Objective:** Use a sales dataset with 750 records to calculate total products sold for each product line.

**Dataset theme:** Specialty craft and hobby products, including miniature painting, bookbinding, model railway, and botanical art supplies.

We use PySpark DataFrame operations:
1. Read the CSV file.
2. Group records by `PRODUCTLINE`.
3. Sum `QUANTITYORDERED`.

In [ ]:
# Install dependency if needed:
# %pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as spark_sum

spark = (
    SparkSession.builder
    .appName("ProductsSoldByCategory")
    .master("local[*]")
    .getOrCreate()
)


In [ ]:
sales_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "sample_sales_data.csv"))
)

sales_spark.printSchema()
sales_spark.show(5)


In [ ]:
products_by_category = (
    sales_spark
    .groupBy("PRODUCTLINE")
    .agg(spark_sum("QUANTITYORDERED").alias("TOTAL_PRODUCTS_SOLD"))
    .orderBy("TOTAL_PRODUCTS_SOLD", ascending=False)
)

products_by_category.show()


In [ ]:
# Convert to pandas for visualization
spark_result_pd = products_by_category.toPandas()

spark_result_pd.plot(
    kind="bar",
    x="PRODUCTLINE",
    y="TOTAL_PRODUCTS_SOLD",
    figsize=(11, 5),
    legend=False,
    title="Total Products Sold by Product Line"
)
plt.xlabel("Product Line")
plt.ylabel("Total Products Sold")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

spark.stop()


**Conclusion:** Spark aggregation groups the sales records by product line and calculates the total quantity ordered for each category efficiently.

## Assignment 4 : Spark Streaming - Fault Tolerance, Checkpointing and Duplicate Detection
Dataset: UCI Online Retail Dataset (541,909 transactions)
Data Source: Online Retail CSV - File Streaming


In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *

pdf = pd.read_excel("Online Retail.xlsx", nrows=5000)

pdf["InvoiceNo"] = pdf["InvoiceNo"].astype(str)
pdf["StockCode"] = pdf["StockCode"].astype(str)
pdf["CustomerID"] = pdf["CustomerID"].astype(str)
pdf["InvoiceDate"] = pd.to_datetime(pdf["InvoiceDate"])

pdf = pdf.dropna(subset=["InvoiceNo", "StockCode", "InvoiceDate"])

input_dir = "data/online_retail_stream"
os.makedirs(input_dir, exist_ok=True)

for i in range(0, len(pdf), 1000):
    batch = pdf.iloc[i:i+1000].copy()
    batch = pd.concat([batch, batch.head(5)])
    batch.to_json(
        f"{input_dir}/batch_{i//1000}.json",
        orient="records", lines=True, date_format="iso"
    )

schema = StructType([
    StructField("InvoiceNo", StringType()),
    StructField("StockCode", StringType()),
    StructField("Description", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("InvoiceDate", TimestampType()),
    StructField("UnitPrice", DoubleType()),
    StructField("CustomerID", StringType()),
    StructField("Country", StringType())
])

spark = SparkSession.builder \
    .appName("OnlineRetailStreaming") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

stream_df = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(input_dir)

clean_df = stream_df \
    .withWatermark("InvoiceDate", "1 day") \
    .dropDuplicates(["InvoiceNo", "StockCode", "InvoiceDate"])

query = clean_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", "data/retail_checkpoint") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()

print("Streaming Completed Successfully")

spark.stop()

## Assignment 5 : LZW - Data Compression and Decompression
Data Source: Sample Text String

In [8]:
def compress(text):
    dictionary = {chr(i): i for i in range(256)}
    next_code = 256
    w = ''
    result = []

    for c in text:
        wc = w + c
        if wc in dictionary:
            w = wc
        else:
            result.append(dictionary[w])
            dictionary[wc] = next_code
            next_code += 1
            w = c

    if w:
        result.append(dictionary[w])
    return result


def decompress(data):
    if not data:
        return ''

    dictionary = {i: chr(i) for i in range(256)}
    next_code = 256
    w = chr(data[0])
    result = w

    for k in data[1:]:
        if k in dictionary:
            entry = dictionary[k]
        elif k == next_code:
            entry = w + w[0]
        else:
            raise ValueError('Invalid compressed data')

        result += entry
        dictionary[next_code] = w + entry[0]
        next_code += 1
        w = entry

    return result


text = 'ABABABAABABA'
compressed = compress(text)
decompressed = decompress(compressed)

print('Original Data:', text)
print('Compressed Data:', compressed)
print('Decompressed Data:', decompressed)
print('Compression Successful:', text == decompressed)

Original Data: ABABABAABABA
Compressed Data: [65, 66, 256, 258, 258, 257]
Decompressed Data: ABABABAABABA
Compression Successful: True
